# Reintegration & recovery — explanatory + predictive pipeline (IS 455 / CRISP-DM)

**Package:** `ml-pipelines/ml_pipeline_reintegration_effectiveness/`  
**Modular scripts:** `data_prep.py`, `feature_engineering.py`, `train_explanatory.py`, `train_predictive.py`, `evaluate.py`, `export_artifacts.py`, `inference_example.py`

## 1. Problem framing
**Business question:** Which factors are **most associated** with **successful reintegration** (`reintegration_status = Completed`), and how can we **prioritize** residents who may need more support (decision support — **not** automated decisions)? **Stakeholders:** case management, clinical leadership, executive sponsor. **Predictive vs explanatory:** **Explanatory** models (logistic) clarify **associations** for staff training and audits. **Predictive** models improve **ranking** for supervision queues. **Ethics:** Staff-only, authenticated use; human override; never remove services based solely on scores.

## 2. Data acquisition, preparation & exploration
Multi-table resident timeline from CSV extracts; `data_prep.prepare_tables` aligns dates and grains; `feature_engineering.build_feature_matrix` assembles explanatory variables. Documented joins follow resident keys and event dates. EDA in-notebook for distributions and missingness. **Reproducible pipeline:** module entrypoints + `run_all.py`.

## 3. Modeling & feature selection
Compare **logistic** (interpretable) vs **tree/ensemble** (predictive) per `train_explanatory` / `train_predictive`. Feature sets justified by domain (placement, services, risk indicators). Hyperparameters in training code.

## 4. Evaluation & interpretation
Holdout metrics (AUC, calibration, confusion-derived costs). **False negative:** resident who needs help is deprioritized → **worse outcomes**. **False positive:** unnecessary intensive case load → **staff burnout** and opportunity cost.

## 5. Causal and relationship analysis
Coefficients are **conditional associations**; unmeasured confounders (family support, off-books services) break causal claims. Importances from ensembles summarize **prediction contribution**, not treatment effects. **Honest stance:** use outputs to **guide questions**, not to assert causality.

## 6. Deployment notes
**Artifacts:** `serialized_models/` + `outputs/`. Refresh via `ml_backend_export` into `App_Data/ml` as configured for this pipeline. **Integration:** .NET ML inference services + admin/resident UI where effectiveness scores are surfaced (see repo `Controllers/Ml*.cs` and scaffold pages referencing resident ML).

## 1. Environment and imports

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for d in [p, *p.parents]:
        if (d / "ml-pipelines").is_dir() and (d / "data" / "lighthouse_csv_v7").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find repo root (need ml-pipelines/ and data/lighthouse_csv_v7/). "
        f"cwd={p}"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
ML_PIPELINES_DIR = REPO_ROOT / "ml-pipelines"
if str(ML_PIPELINES_DIR) not in sys.path:
    sys.path.insert(0, str(ML_PIPELINES_DIR))

from ml_pipeline_reintegration_effectiveness import config
from ml_pipeline_reintegration_effectiveness.data_prep import prepare_tables
from ml_pipeline_reintegration_effectiveness.feature_engineering import build_feature_matrix, build_label
from ml_pipeline_reintegration_effectiveness.export_artifacts import export_from_trained_bundle
from ml_pipeline_reintegration_effectiveness.train_explanatory import run_explanatory, split_xy
from ml_pipeline_reintegration_effectiveness.train_predictive import train_predictive_full
from ml_pipeline_reintegration_effectiveness.evaluate import evaluate_pipeline, feature_importance_dataframe

config.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
config.SERIALIZED_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", config.DEFAULT_DATA_DIR)

## 2. Load tables and define observation date

In [ ]:
tables, obs_date = prepare_tables(config.DEFAULT_DATA_DIR)
res = tables["residents"]
print("observation_date:", obs_date)
print(res[["reintegration_status", "case_status"]].value_counts())

## 3. Target distribution and censoring note

In [ ]:
y_raw = build_label(res)
print("Completed rate:", y_raw.mean())
print(y_raw.value_counts())
# Active + In Progress coded 0 — not necessarily failure (censored).
print(res.loc[res["case_status"].eq("Active"), "reintegration_status"].value_counts())

## 4. Build leakage-safe feature matrix

In [ ]:
X, y, fe_meta = build_feature_matrix(tables, obs_date)
print("X shape:", X.shape)
print(json.dumps({k: fe_meta[k] for k in ("window_start", "window_end", "gap_days", "label_definition")}, indent=2))
X.head()

## 5. EDA — missingness, numeric distributions, correlations

In [ ]:
num_cols = fe_meta["numeric_features"]
miss = X[num_cols].isna().mean().sort_values(ascending=False)
print("Top missing rates:\n", miss.head(10))

fig, ax = plt.subplots(figsize=(6, 4))
y.astype(int).value_counts().sort_index().plot(kind="bar", ax=ax, color=["steelblue", "darkorange"])
ax.set_xticklabels(["Not Completed (incl. censored)", "Completed"], rotation=0)
ax.set_title("Target distribution")
ax.set_ylabel("Count")
plt.tight_layout()
fig.savefig(config.OUTPUTS_DIR / "fig_target_distribution.png", dpi=120)
plt.show()

sample_num = [c for c in ["days_in_program_at_window_end", "pr_progress_rate", "hv_favorable_rate", "edu_mean_progress"] if c in num_cols]
X[sample_num].hist(figsize=(8, 6), bins=15)
plt.suptitle("Example numeric features")
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "fig_numeric_histograms.png", dpi=120)
plt.show()

corr = X[num_cols].corr(numeric_only=True)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="vlag", center=0)
plt.title("Numeric feature correlation matrix")
plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "fig_correlation_matrix.png", dpi=120)
plt.show()

## 6. Explanatory model (training split only — aligned with predictive train)

In [ ]:
bundle = train_predictive_full(X, y, fe_meta)
X_tr, y_tr = bundle["X_train"], bundle["y_train"]
expl_pipe, coef_df = run_explanatory(X_tr, y_tr, fe_meta, outputs_dir=config.OUTPUTS_DIR)
display(coef_df.head(20))

## 7. Predictive models, CV selection, holdout evaluation

In [ ]:
pipe = bundle["pipeline"]
_, X_te = split_xy(bundle["X_test"], fe_meta["numeric_features"], fe_meta["categorical_features"])
metrics = evaluate_pipeline(pipe, X_te, bundle["y_test"])
print(json.dumps(metrics, indent=2))
print("\nCV model comparison:")
display(bundle["cv_results"])
print("Tuning info:", bundle["tuning"])

## 8. Feature importance (predictive model, if available)

In [ ]:
imp = feature_importance_dataframe(pipe)
if imp is not None:
    display(imp.head(15))
    plt.figure(figsize=(8, 5))
    imp.head(15).sort_values("importance").plot.barh(x="feature", y="importance", legend=False)
    plt.title("Top 15 importances")
    plt.tight_layout()
    plt.savefig(config.OUTPUTS_DIR / "fig_top_importances.png", dpi=120)
    plt.show()
else:
    print("Final model has no tree feature_importances_ (e.g. logistic).")

## 9. Export joblib + JSON artifacts for web app

In [ ]:
info = export_from_trained_bundle(bundle, X, y, fe_meta)
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in info.items()}, indent=2))

## 10. Optional — re-run inference CLI\n\n`python -m ml_pipeline_reintegration_effectiveness.inference_example` from `ml-pipelines/`.